In [1]:
import pandas as pd
import numpy as np

In [2]:
from datasets import Dataset
from sklearn.model_selection import train_test_split

# Preparar datos y divisiones de train y test
df = pd.read_csv('DB/youtube.csv')


c:\Program Files\Python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Limpieza

In [3]:
df.drop_duplicates(inplace=True)
df.drop(columns=['link'], inplace=True)
df.drop(columns=['description'], inplace=True)
df["title"] = df["title"].astype(str).str.strip()

In [4]:
display(df)

,title,category
0,Ep 1| Travelling through North East India | Of...,travel
1,Welcome to Bali | Travel Vlog | Priscilla Lee,travel
2,My Solo Trip to ALASKA | Cruising From Vancouv...,travel
3,Traveling to the Happiest Country in the World!!,travel
4,Solo in Paro Bhutan | Tiger's Nest visit | Bhu...,travel
...,...,...
3594,21st Century Challenges: Crash Course European...,history
3595,EU DataViz webinar - Barnaby Skinner - How to ...,history
3596,Stone Age Scandinavia: First People In the Nor...,history
3597,AP European History - Interwar Period: Paris P...,history


# Dividir (Entrenamiento/ Prueba)

In [5]:
classes = sorted(df["category"].astype(str).unique().tolist())
label2id = {c:i for i, c in enumerate(classes)}
id2label = {i:c for c, i in label2id.items()}

# Crear columna 'label' numérica a partir de 'category'
df = df.copy()
df["label"] = df["category"].astype(str).map(label2id)

In [6]:

train_df, val_df = train_test_split(
    df[["title","label"]],
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds   = Dataset.from_pandas(val_df,   preserve_index=False)

In [7]:
train_ds

Dataset({
    features: ['title', 'label'],
    num_rows: 2800
})

In [8]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Modelo y tokenizador
model_id = "bert-base-cased"
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=4, id2label=id2label, label2id=label2id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
from transformers import DataCollatorWithPadding

# Rellenar hasta la secuencia más larga.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def preprocess_function(examples):
   """Tokenize datos de entrada"""
   return tokenizer(examples["title"], truncation=True)

# Tokenize train/test
tokenized_train = train_ds.map(preprocess_function, batched=True)
tokenized_test = val_ds.map(preprocess_function, batched=True)

Map:   0%|          | 0/2800 [00:00<?, ? examples/s]

Map: 100%|██████████| 700/700 [00:00<00:00, 36287.84 examples/s]


In [10]:
import numpy as np
import evaluate

def compute_metrics(eval_pred):
  """Calcular F1 score"""
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=-1)

  load_f1 = evaluate.load("f1")
  f1 = load_f1.compute(predictions=predictions, references=labels, average="macro")["f1"]
  return {"f1": f1}

In [11]:
from transformers import TrainingArguments, Trainer

# Argumentos de entrenamiento para el ajuste de parámetros
training_args = TrainingArguments(
   "model",
   learning_rate=2e-5,
   per_device_train_batch_size=8,
   per_device_eval_batch_size=8,
   num_train_epochs=1,
   weight_decay=0.01,
   save_strategy="epoch",
   report_to="none"
)

# "trainer" ejecuta el proceso de entrenamiento
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   tokenizer=tokenizer,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)

C:\Users\death\AppData\Local\Temp\ipykernel_25144\2664990676.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [12]:
trainer.train()

C:\Users\death\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


TrainOutput(global_step=350, training_loss=0.3152236066545759, metrics={'train_runtime': 301.7328, 'train_samples_per_second': 9.28, 'train_steps_per_second': 1.16, 'total_flos': 53544061249536.0, 'train_loss': 0.3152236066545759, 'epoch': 1.0})

In [13]:
trainer.evaluate()

C:\Users\death\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.20176023244857788,
 'eval_f1': 0.9538888592497747,
 'eval_runtime': 19.0643,
 'eval_samples_per_second': 36.718,
 'eval_steps_per_second': 4.616,
 'epoch': 1.0}

In [14]:
trainer.save_model("model2")
tokenizer.save_pretrained("tokenizer2")

('tokenizer2\\tokenizer_config.json',
 'tokenizer2\\special_tokens_map.json',
 'tokenizer2\\vocab.txt',
 'tokenizer2\\added_tokens.json',
 'tokenizer2\\tokenizer.json')

In [15]:
loaded_tokenizer = AutoTokenizer.from_pretrained("tokenizer2")
loaded_model = AutoModelForSequenceClassification.from_pretrained("model2")

In [17]:
import evaluate
import torch

def predict(texts):
    enc = loaded_tokenizer(texts, truncation=True, padding=True, max_length=256, return_tensors="pt")
    with torch.no_grad():
        out = loaded_model(**{k: v.to(loaded_model.device) for k, v in enc.items()})
        pred_ids = out.logits.argmax(dim=-1).cpu().numpy().tolist()
    return [loaded_model.config.id2label[i] for i in pred_ids]

print(predict([
    "I loved the museum and the old town tour.",
    "The tacos were amazing and fresh!",
    "El concierto de anoche fue espectacular, la banda tocó todos sus éxitos y el público estaba encantado.",
    "Viaje por Europa en 15 días, visitando 5 países y 10 ciudades, disfrutando de la cultura y la gastronomía local.",
    "Repasando la independcia de mexico y sus causas"
]))

['travel', 'food', 'travel', 'travel', 'history']
